# Classical Data Science Example: Titanic Survival Prediction

This notebook is designed as a workshop-friendly baseline for agent-driven software development with data scientists.

The goal is deliberately classical:

> Given passenger information from the Titanic dataset, predict whether a passenger survived.

The notebook follows a standard data science workflow:

1. Load data
2. Explore the dataset
3. Clean and transform features
4. Train baseline models
5. Evaluate results
6. Inspect feature importance
7. Save a trained model artifact

This is a good starting point for agentic workflows because it can later be refactored into scripts, tests, a package, or a small app.

## 1. Setup

This notebook uses common Python data science libraries.

If needed, install missing packages with:

```bash
pip install pandas numpy matplotlib scikit-learn joblib pytest
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

## 2. Load the Titanic dataset

We load a simple public CSV version of the Titanic dataset.

The target column is `survived`:

- `0`: did not survive
- `1`: survived

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(DATA_URL)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 3. Basic exploratory data analysis

Start with the target distribution.

In [ ]:
df["survived"].value_counts(normalize=True).rename("survival_rate")

In [ ]:
ax = df["survived"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Survival counts")
ax.set_xlabel("Survived")
ax.set_ylabel("Passenger count")
plt.show()

Look at a few relationships that are usually predictive in this dataset.

In [ ]:
df.groupby("sex")["survived"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("class")["survived"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("alone")["survived"].mean().sort_values(ascending=False)

In [ ]:
ax = df.boxplot(column="age", by="survived")
ax.set_title("Age distribution by survival")
ax.set_xlabel("Survived")
ax.set_ylabel("Age")
plt.suptitle("")
plt.show()

## 4. Select features

For the first version, keep the feature set intentionally simple.

This is a useful workshop choice because participants can later ask an agent to improve it.

In [ ]:
target = "survived"

numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "class", "embarked", "alone"]

features = numeric_features + categorical_features

X = df[features].copy()
X["alone"] = X["alone"].astype(str)
y = df[target]

X.head()

## 5. Train/test split

We use a holdout test set to estimate performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

## 6. Build preprocessing pipelines

Classical machine learning models need clean numeric inputs.

We will:

- Impute missing numeric values with the median
- Scale numeric values
- Convert categorical values to strings
- Impute missing categorical values with a fixed `missing` label
- One-hot encode categorical values

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 7. Train baseline models

We compare three models:

1. Dummy classifier: a simple baseline
2. Logistic regression: interpretable classical model
3. Random forest: stronger nonlinear model

In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=3,
    ),
}

results = []

for name, model in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_score)
    else:
        roc_auc = np.nan

    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "roc_auc": roc_auc,
        }
    )

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
results_df

## 8. Cross-validation

A single train/test split can be noisy. Cross-validation gives a more stable estimate.

In [ ]:
cv_results = []

for name, model in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=5,
        scoring="roc_auc",
    )

    cv_results.append(
        {
            "model": name,
            "mean_roc_auc": scores.mean(),
            "std_roc_auc": scores.std(),
        }
    )

cv_results_df = pd.DataFrame(cv_results).sort_values("mean_roc_auc", ascending=False)
cv_results_df

## 9. Evaluate the selected model

For this baseline, select the random forest model.

In [ ]:
best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            min_samples_leaf=3,
        )),
    ]
)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", round(roc_auc_score(y_test, y_score), 3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion matrix")
plt.show()

## 10. Inspect feature importance

For tree-based models, we can inspect which transformed features were most important.

In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
)

importance_df.head(15)

In [ ]:
top_n = 15
ax = importance_df.head(top_n).sort_values("importance").plot(
    kind="barh",
    x="feature",
    y="importance",
    legend=False,
)
ax.set_title(f"Top {top_n} feature importances")
ax.set_xlabel("Importance")
ax.set_ylabel("")
plt.show()

## 11. Try predictions on example passengers

This is useful for explaining how the model behaves.

In [ ]:
example_passengers = pd.DataFrame(
    [
        {
            "age": 30,
            "fare": 50,
            "sibsp": 0,
            "parch": 0,
            "sex": "female",
            "class": "First",
            "embarked": "S",
            "alone": "True",
        },
        {
            "age": 30,
            "fare": 8,
            "sibsp": 0,
            "parch": 0,
            "sex": "male",
            "class": "Third",
            "embarked": "S",
            "alone": "True",
        },
    ]
)

survival_probability = best_model.predict_proba(example_passengers)[:, 1]

example_passengers.assign(
    predicted_survival_probability=survival_probability.round(3)
)

## 12. Save the trained model

This creates a reusable model artifact.

In an agent-driven software development workshop, this is a natural point to ask an agent to:

- Move preprocessing and training into Python modules
- Add a CLI
- Add tests
- Add model metadata
- Add a simple prediction API

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

model_path = ARTIFACT_DIR / "titanic_survival_model.joblib"
joblib.dump(best_model, model_path)

model_path